# Combine the taxi, POI, and weather data 

In [17]:
import pandas as pd
import numpy as np
import geopandas as gpd
import holidays

## Read the data

In [18]:
# weather data
weather_df = pd.read_parquet("../data/weather_data.parquet")
weather_df.head()

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491


In [19]:
# POI data
poi_df = gpd.read_file("../data/chicago_pois_cleaned.gpkg")
poi_df.head()

,id,poi_type,name,h3_index,geometry
0,20217109,ferry_terminal,Shoreline Sightseeing,882664c1e3fffff,POINT (-87.62252 41.88914)
1,20217442,ferry_terminal,Union Station/Willis Tower,882664c1adfffff,POINT (-87.63774 41.87906)
2,271275603,theatre,Storefront Theater,882664c1a9fffff,POINT (-87.62551 41.88481)
3,282370094,convenience,7-Eleven,88275936b1fffff,POINT (-87.8373 41.97632)
4,286372562,fast_food,Chipotle,882664c107fffff,POINT (-87.64921 41.92557)


In [20]:
# taxi data
taxi_df = pd.read_parquet("../data/taxi_data_processed.parquet") 
taxi_df.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,h3_index_pickup,h3_index_dropoff,Pickup Hour,Dropoff Hour
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,882664cad7fffff,882664cad7fffff,2026-04-30 23:00:00,2026-05-01
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,882664cad7fffff,882664c1e5fffff,2026-04-30 23:00:00,2026-05-01
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,882664cad7fffff,882664c1abfffff,2026-04-30 23:00:00,2026-05-01
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,88275934edfffff,882664c1abfffff,2026-04-30 23:00:00,2026-05-01
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,88275934edfffff,882664cad7fffff,2026-04-30 23:00:00,2026-05-01


## Merge the Data

In [21]:
# join weather and taxi data upon the time as key 
taxi_weather_df = taxi_df.merge(
    weather_df,
    left_on="Pickup Hour",
    right_on="time_step",
    how="inner"
)

taxi_weather_df.head()

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,...,Dropoff Hour,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,...,2026-05-01,2026-04-30 23:00:00,279.45874,0.001396,0.0,0.0,-3.47055,-1.344467,6.30874,1.396
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,...,2026-05-01,2026-04-30 23:00:00,279.45874,0.001396,0.0,0.0,-3.47055,-1.344467,6.30874,1.396
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,...,2026-05-01,2026-04-30 23:00:00,279.45874,0.001396,0.0,0.0,-3.47055,-1.344467,6.30874,1.396
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,...,2026-05-01,2026-04-30 23:00:00,279.45874,0.001396,0.0,0.0,-3.47055,-1.344467,6.30874,1.396
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,...,2026-05-01,2026-04-30 23:00:00,279.45874,0.001396,0.0,0.0,-3.47055,-1.344467,6.30874,1.396


## Feature Engineering

In [22]:
# combine the raw wind data into the wind speed and direction
taxi_weather_df["wind_speed"] = (taxi_weather_df["u10"]**2 + taxi_weather_df["v10"]**2) ** 0.5
taxi_weather_df["wind_dir"] = (270 - np.degrees(np.arctan2(taxi_weather_df["u10"], taxi_weather_df["v10"]))*180/np.pi)%360

taxi_weather_df

,time_step,2m_temp,total_precip,snow_cov,snow_depth,u10,v10,2m_temp_c,total_precip_mm
0,2024-01-01 00:00:00,274.39615,0.000281,6.929688,0.007812,2.705109,-5.971687,1.24615,0.280723
1,2024-01-01 01:00:00,274.24170,0.000150,8.179688,0.008789,2.658508,-7.036841,1.09170,0.150489
2,2024-01-01 02:00:00,274.05762,0.000030,8.804688,0.008789,2.478607,-7.009237,0.90762,0.030188
3,2024-01-01 03:00:00,273.90906,0.000014,8.873047,0.008789,2.442825,-6.861008,0.75906,0.013527
4,2024-01-01 04:00:00,274.01672,0.000037,8.890625,0.008789,2.084305,-6.889282,0.86672,0.037491
...,...,...,...,...,...,...,...,...,...
20707,2026-05-12 19:00:00,293.30435,0.000342,0.000000,0.000000,1.670914,4.648880,20.15435,0.341564
20708,2026-05-12 20:00:00,293.64343,0.000003,0.000000,0.000000,1.859116,3.183975,20.49343,0.002533
20709,2026-05-12 21:00:00,293.37470,0.000053,0.000000,0.000000,2.806747,3.596573,20.22470,0.052869
20710,2026-05-12 22:00:00,292.62510,0.000749,0.000000,0.000000,3.238815,4.192703,19.47510,0.748754


In [23]:
# split the date into several parts
taxi_weather_df['hour'] = taxi_weather_df['time_step'].dt.hour
taxi_weather_df['day'] = taxi_weather_df['time_step'].dt.day
taxi_weather_df['month'] = taxi_weather_df['time_step'].dt.month
taxi_weather_df['weekday'] = taxi_weather_df['time_step'].dt.weekday

taxi_weather_df

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,...,u10,v10,2m_temp_c,total_precip_mm,wind_speed,wind_dir,hour,day,month,weekday
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,...,-3.470550,-1.344467,6.30874,1.396000,3.721870,159.916863,23,30,4,3
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,...,-3.470550,-1.344467,6.30874,1.396000,3.721870,159.916863,23,30,4,3
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,...,-3.470550,-1.344467,6.30874,1.396000,3.721870,159.916863,23,30,4,3
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,...,-3.470550,-1.344467,6.30874,1.396000,3.721870,159.916863,23,30,4,3
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,...,-3.470550,-1.344467,6.30874,1.396000,3.721870,159.916863,23,30,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6012344,a75082c74bf8ae2f97d9811101952ba5de8192aa,50fcee6711df1d794e4f337c99f44abe8109795ec69474...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,452.000,0.20,17031081700,17031081700,575.0,Medallion Leasin,...,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0
6012345,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,180.000,0.30,17031081500,17031081201,525.0,"Taxicab Insurance Agency, LLC",...,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0
6012346,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,01/01/2024 12:00:00 AM,01/01/2024 12:30:00 AM,1.681,15.34,17031980000,17031071400,5310.0,Globe Taxi,...,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0
6012347,ddcd4d6b7c138bee6841a7800cfbb45f31e6101a,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,01/01/2024 12:00:00 AM,01/01/2024 12:45:00 AM,3.059,17.44,17031980000,17031320100,6630.0,City Service,...,2.705109,-5.971687,1.24615,0.280723,6.555811,353.058005,0,1,1,0


In [24]:
taxi_weather_df['hour_sin'] = np.sin(2 * np.pi * taxi_weather_df['hour'] / 24)
taxi_weather_df['hour_cos'] = np.cos(2 * np.pi * taxi_weather_df['hour'] / 24)

taxi_weather_df['month_sin'] = np.sin(2 * np.pi * taxi_weather_df['month'] / 12)
taxi_weather_df['month_cos'] = np.cos(2 * np.pi * taxi_weather_df['month'] / 12)

taxi_weather_df

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,...,wind_speed,wind_dir,hour,day,month,weekday,hour_sin,hour_cos,month_sin,month_cos
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,...,3.721870,159.916863,23,30,4,3,-0.258819,0.965926,0.866025,-0.500000
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,...,3.721870,159.916863,23,30,4,3,-0.258819,0.965926,0.866025,-0.500000
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,...,3.721870,159.916863,23,30,4,3,-0.258819,0.965926,0.866025,-0.500000
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,...,3.721870,159.916863,23,30,4,3,-0.258819,0.965926,0.866025,-0.500000
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,...,3.721870,159.916863,23,30,4,3,-0.258819,0.965926,0.866025,-0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6012344,a75082c74bf8ae2f97d9811101952ba5de8192aa,50fcee6711df1d794e4f337c99f44abe8109795ec69474...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,452.000,0.20,17031081700,17031081700,575.0,Medallion Leasin,...,6.555811,353.058005,0,1,1,0,0.000000,1.000000,0.500000,0.866025
6012345,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,180.000,0.30,17031081500,17031081201,525.0,"Taxicab Insurance Agency, LLC",...,6.555811,353.058005,0,1,1,0,0.000000,1.000000,0.500000,0.866025
6012346,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,01/01/2024 12:00:00 AM,01/01/2024 12:30:00 AM,1.681,15.34,17031980000,17031071400,5310.0,Globe Taxi,...,6.555811,353.058005,0,1,1,0,0.000000,1.000000,0.500000,0.866025
6012347,ddcd4d6b7c138bee6841a7800cfbb45f31e6101a,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,01/01/2024 12:00:00 AM,01/01/2024 12:45:00 AM,3.059,17.44,17031980000,17031320100,6630.0,City Service,...,6.555811,353.058005,0,1,1,0,0.000000,1.000000,0.500000,0.866025


In [25]:
taxi_weather_df["season"] = taxi_weather_df["Pickup Hour"].dt.month % 12 // 3 + 1
# Mapping: 1 = winter, 2 = spring, 3 = summer, 4 = autumn

taxi_weather_df["season_sin"] = np.sin(2 * np.pi * taxi_weather_df["season"] / 4)
taxi_weather_df["season_cos"] = np.cos(2 * np.pi * taxi_weather_df["season"] / 4)

taxi_weather_df

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,...,day,month,weekday,hour_sin,hour_cos,month_sin,month_cos,season,season_sin,season_cos
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,...,30,4,3,-0.258819,0.965926,0.866025,-0.5,2,1.224647e-16,-1.0
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,...,30,4,3,-0.258819,0.965926,0.866025,-0.5,2,1.224647e-16,-1.0
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,...,30,4,3,-0.258819,0.965926,0.866025,-0.5,2,1.224647e-16,-1.0
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,...,30,4,3,-0.258819,0.965926,0.866025,-0.5,2,1.224647e-16,-1.0
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,...,30,4,3,-0.258819,0.965926,0.866025,-0.5,2,1.224647e-16,-1.0


In [26]:
# introduce holidays as feature
us_holidays = holidays.US(state="IL")

taxi_weather_df["date"] = taxi_weather_df["time_step"].dt.date
taxi_weather_df["is_holiday"] = taxi_weather_df["date"].isin(us_holidays).astype(int)

# also flag dates that are close to a holiday
taxi_weather_df["is_near_holiday"] = (
    taxi_weather_df["date"].isin(us_holidays) |
    taxi_weather_df["date"].shift(1).isin(us_holidays) |
    taxi_weather_df["date"].shift(-1).isin(us_holidays)
).astype(int)

taxi_weather_df

,Trip ID,Taxi ID,Trip Start Timestamp,Trip End Timestamp,Trip Seconds,Trip Miles,Pickup Census Tract,Dropoff Census Tract,Trip Total,Company,...,hour_sin,hour_cos,month_sin,month_cos,season,season_sin,season_cos,date,is_holiday,is_near_holiday
0,a1e30fb944a6fd2598a9c3caf0192121f0e39dea,249d932165de24c10140f979fa81b350183cff7ac63099...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,261.000,0.40,17031833000,17031833000,500.0,Globe Taxi,...,-0.258819,0.965926,0.866025,-0.500000,2,1.224647e-16,-1.000000e+00,2026-04-30,0,0
1,07df4fb41ec699a5287f8a4e64726563bae385fc,fed019ef14a3bfeaa5ef5f523be5d8dc28943952aa66a3...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,240.000,0.90,17031833000,17031081800,525.0,Choice Taxi Association Inc,...,-0.258819,0.965926,0.866025,-0.500000,2,1.224647e-16,-1.000000e+00,2026-04-30,0,0
2,5d9e23fa8a4beb9d495f5c4c7c00bf33026b561e,fb1ed566274f6a66dfcddbe4c014bbe87ac3516412dfe6...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,425.000,1.41,17031833000,17031320400,845.0,Sun Taxi,...,-0.258819,0.965926,0.866025,-0.500000,2,1.224647e-16,-1.000000e+00,2026-04-30,0,0
3,e6ba74242fb88764fb6cb5e34a7f399b613c1a6b,3278a0dd17ed3de47adf8cf7236bf8f21f29e62c04394b...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.560,16.80,17031980000,17031320400,5375.0,Transit Administrative Center Inc,...,-0.258819,0.965926,0.866025,-0.500000,2,1.224647e-16,-1.000000e+00,2026-04-30,0,0
4,876f1c50d59ac611ff5daca49934a046c951de40,3f365e125eb1d15aae0ae8a4bf53a01191478f26ccf00c...,04/30/2026 11:45:00 PM,05/01/2026 12:00:00 AM,1.297,17.10,17031980000,17031833000,4725.0,5 Star Taxi,...,-0.258819,0.965926,0.866025,-0.500000,2,1.224647e-16,-1.000000e+00,2026-04-30,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6012344,a75082c74bf8ae2f97d9811101952ba5de8192aa,50fcee6711df1d794e4f337c99f44abe8109795ec69474...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,452.000,0.20,17031081700,17031081700,575.0,Medallion Leasin,...,0.000000,1.000000,0.500000,0.866025,1,1.000000e+00,6.123234e-17,2024-01-01,0,0
6012345,63d8c865c01bde9e17e469db6a30e33c8cfe5314,259d38cfdbc9ac6f9bb01f0df740e0ddf4a631a70bbdd6...,01/01/2024 12:00:00 AM,01/01/2024 12:00:00 AM,180.000,0.30,17031081500,17031081201,525.0,"Taxicab Insurance Agency, LLC",...,0.000000,1.000000,0.500000,0.866025,1,1.000000e+00,6.123234e-17,2024-01-01,0,0
6012346,3c05ccf0732fc338b7c875f9a9779039eaada274,0cbf5c0f6aca3628d77c7b6fe89715757ed402a70b0f8b...,01/01/2024 12:00:00 AM,01/01/2024 12:30:00 AM,1.681,15.34,17031980000,17031071400,5310.0,Globe Taxi,...,0.000000,1.000000,0.500000,0.866025,1,1.000000e+00,6.123234e-17,2024-01-01,0,0
6012347,ddcd4d6b7c138bee6841a7800cfbb45f31e6101a,0fdab9be71f6d88e3d3a2e115afc5a33d2bf74153792c5...,01/01/2024 12:00:00 AM,01/01/2024 12:45:00 AM,3.059,17.44,17031980000,17031320100,6630.0,City Service,...,0.000000,1.000000,0.500000,0.866025,1,1.000000e+00,6.123234e-17,2024-01-01,0,0
